# Session 19 Solutions: Grouping and Summarising

Data Science with Python, Level 1, Stage 2: NumPy and pandas

Try each exercise yourself before reading its solution. There is usually more than one correct answer; compare your approach with this one.

## Setup

In [1]:
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
df = pd.read_csv(DATA_URL + "cafe_sales.csv")
df["revenue"] = df["unit_price"] * df["quantity"]
df["date"] = pd.to_datetime(df["date"])

## In-Class Exercises

### In-Class Exercise 1: Revenue per Item

In [2]:
df.groupby("item")["revenue"].sum().sort_values(ascending=False).head(5)

item
Chicken Sandwich    1014.0
Latte                885.5
Cappuccino           700.0
Veggie Sandwich      478.5
Espresso             477.5
Name: revenue, dtype: float64

### In-Class Exercise 2: Branch Summary

In [3]:
df.groupby("branch").agg(orders=("order_id", "count"), avg_qty=("quantity", "mean"),
                         items=("item", "nunique"), revenue=("revenue", "sum")).round(2)

,orders,avg_qty,items,revenue
branch,,,,
Campus,355,1.69,12,2041.5
Downtown,410,1.66,12,2374.2
Mall,235,1.61,12,1307.4


### In-Class Exercise 3: Category by Hour

In [4]:
pd.pivot_table(df, index="hour", columns="category", values="order_id", aggfunc="count", fill_value=0)

category,Coffee,Juice,Pastry,Sandwich,Tea
hour,,,,,
7,27,7,13,8,3
8,54,7,25,8,13
9,50,8,19,21,15
10,45,5,10,19,10
11,29,5,16,6,4
12,27,6,17,7,7
13,40,6,17,11,12
14,36,4,17,9,9
15,30,3,19,12,9


### In-Class Exercise 4: Share of Branch

In [5]:
df["share"] = df["revenue"] / df.groupby("branch")["revenue"].transform("sum")
print(df.groupby("branch")["share"].sum().round(6))

branch
Campus      1.0
Downtown    1.0
Mall        1.0
Name: share, dtype: float64


## Part A: Basic

### Exercise 1: Average GPA per Major

In [6]:
students = pd.read_csv(DATA_URL + "students.csv")
students.groupby("major").agg(avg_gpa=("gpa", "mean"), students=("name", "count")).round(2).sort_values("avg_gpa", ascending=False)

,avg_gpa,students
major,,
Design,3.16,5
Business,3.08,7
Engineering,3.06,5
Computer Science,3.01,7
Medicine,2.67,6


### Exercise 2: Salaries by Department

In [7]:
emp = pd.read_csv(DATA_URL + "employees.csv")
by_dept = emp.groupby("department")["monthly_salary"].agg(["mean", "min", "max"]).round(0)
print(by_dept)
print("Highest:", by_dept["mean"].idxmax())

               mean   min   max
department                     
Data         2641.0  1690  4510
Engineering  2760.0  1700  4940
Finance      2176.0  1380  3540
Marketing    1999.0  1190  3040
Sales        2111.0  1240  3790
Support      1586.0   800  3870
Highest: Engineering


## Part B: Intermediate

### Exercise 3: Department and Level

In [8]:
emp = pd.read_csv(DATA_URL + "employees.csv")
pt = pd.pivot_table(emp, index="department", columns="level", values="monthly_salary", aggfunc="mean").round(0)
pt = pt[["Junior", "Mid", "Senior", "Lead"]]
print(pt)
print("Biggest step:", (pt["Lead"] - pt["Junior"]).idxmax())

level        Junior     Mid  Senior    Lead
department                                 
Data         2192.0  2407.0  3258.0  3120.0
Engineering  2420.0  2564.0  3149.0  4007.0
Finance      1873.0  2143.0  2656.0  2160.0
Marketing    1623.0  1846.0  2405.0  2670.0
Sales        1795.0  1941.0  2554.0  3442.0
Support      1310.0  1526.0  2011.0  3025.0
Biggest step: Support


### Exercise 4: Weather by Month

In [9]:
w = pd.read_csv(DATA_URL + "weather_2024.csv")
w["date"] = pd.to_datetime(w["date"])
w["month"] = w["date"].dt.month
temp = w.groupby(["month", "city"])["temp_max"].mean().unstack().round(1)
rain = w.groupby(["month", "city"])["rain_mm"].sum().unstack().round(0)
print(temp)
print(rain.idxmax())

city   Mountain Town  Port City
month                          
1                7.6       17.5
2                9.2       17.7
3               11.8       20.7
4               16.1       23.7
5               22.0       26.8
6               25.6       30.0
7               27.2       30.4
8               25.5       29.3
9               21.5       27.4
10              16.9       23.7
11              11.9       21.0
12               8.3       18.9
city
Mountain Town    1
Port City        2
dtype: int32


## Part C: Challenge

### Exercise 5: Above the Department Average

`assign` adds a column to a copy of the table, which is handy inside a chain of methods.

In [10]:
emp = pd.read_csv(DATA_URL + "employees.csv")
emp["vs_group"] = emp["monthly_salary"] - emp.groupby(["department", "level"])["monthly_salary"].transform("mean")
print(emp.sort_values("vs_group", ascending=False).head(5)[["name", "department", "level", "vs_group"]].round(0))
print(emp.assign(above=emp["vs_group"] > 0).groupby("city")["above"].mean().round(2))

              name   department   level  vs_group
216   Ruba Sabbagh  Engineering  Senior    1341.0
235   Sofia Nassar         Data     Mid    1263.0
121  Basel Mansour         Data  Senior    1252.0
179    Fadi Kassem  Engineering     Mid    1186.0
372    Noah Kassem  Engineering     Mid    1146.0
city
Amman     0.29
Beirut    0.11
Cairo     0.02
Dubai     0.98
Name: above, dtype: float64


## Part D: Cumulative Project: Cafe Sales Analysis

### Exercise 6: Monthly Cafe Report

In [11]:
df["month"] = df["date"].dt.month
print(df.groupby("month").agg(orders=("order_id", "count"), revenue=("revenue", "sum"),
                              avg_order=("revenue", "mean")).round(2))
print(pd.pivot_table(df, index="month", columns="branch", values="revenue", aggfunc="sum", margins=True).round(1))
by_item = df.groupby(["month", "item"])["revenue"].sum().sort_values(ascending=False)
print(by_item.groupby(level=0).head(1))
print(pd.crosstab(df["branch"], df["payment"], normalize="index").round(2))

       orders  revenue  avg_order
month                            
1         348   1999.2       5.74
2         342   1903.6       5.57
3         310   1820.3       5.87
branch  Campus  Downtown    Mall     All
month                                   
1        714.5     827.9   456.8  1999.2
2        683.1     773.1   447.4  1903.6
3        643.9     773.2   403.2  1820.3
All     2041.5    2374.2  1307.4  5723.1
month  item            
1      Chicken Sandwich    416.0
2      Chicken Sandwich    331.5
3      Latte               280.0
Name: revenue, dtype: float64
payment   Card  Cash  Mobile
branch                      
Campus    0.57  0.29    0.13
Downtown  0.53  0.31    0.16
Mall      0.60  0.31    0.10


## Part E: Quiz Answers

| Question | Answer | Explanation |
| --- | --- | --- |
| 1 | B | It splits by city and averages the sales in each group. |
| 2 | C | `transform` returns a result as long as the original table. |
| 3 | B | The inner level becomes the column headers. |
| 4 | B | Rows come from `a`, columns from `b`, and each cell sums `c`. |
| 5 | B | `reset_index` moves the index into a column. |